# Anatomy of a Flop — Valori lipsa

## Scopul
Curățăm datele și le pregătim pentru modelare.

### Pași:
1. Eliminarea valorilor lipsă
2. Feature engineering
3. Salvarea datelor curate

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/movies_raw.csv')
print(f" Date încărcate: {df.shape}")

 Date încărcate: (1000, 18)


In [2]:
# Verificăm valorile lipsă înainte de curățare
missing_values = df.isnull().sum().sort_values(ascending=False)

print("Valori lipsă pe coloane:")

if missing_values[missing_values > 0].empty:
    print("Nu există valori lipsă în datasetul brut.")
else:
    print(missing_values[missing_values > 0])

Valori lipsă pe coloane:
Nu există valori lipsă în datasetul brut.


In [3]:
# Exercițiu: simulăm valori lipsă pentru variabila runtime
df_missing_demo = df.copy()

np.random.seed(42)
missing_indices = df_missing_demo.sample(frac=0.05, random_state=42).index

df_missing_demo.loc[missing_indices, 'runtime'] = np.nan

print("Valori lipsă simulate pentru runtime:")
print(df_missing_demo['runtime'].isnull().sum())

Valori lipsă simulate pentru runtime:
50


In [4]:
# Imputăm valorile lipsă simulate folosind mediana
runtime_median = df_missing_demo['runtime'].median()

df_missing_demo['runtime'] = df_missing_demo['runtime'].fillna(runtime_median)

print("Valori lipsă după imputare:")
print(df_missing_demo['runtime'].isnull().sum())

Valori lipsă după imputare:
0


### Interpretarea imputării valorilor lipsă

Pentru a demonstra tratarea valorilor lipsă, am simulat 50 de valori lipsă în variabila `runtime`. După această simulare, coloana avea 50 de observații fără valoare.

Am ales imputarea cu **mediana** deoarece `runtime` poate conține valori extreme, de exemplu filme foarte scurte sau filme foarte lungi. În astfel de situații, media ar putea fi influențată de outlieri și ar putea introduce o valoare mai puțin reprezentativă pentru majoritatea filmelor.

Mediana este o alegere mai robustă, deoarece reprezintă valoarea centrală a distribuției și nu este afectată puternic de extreme. Astfel, completarea valorilor lipsă cu mediana păstrează structura generală a datelor fără să distorsioneze variabila.

După imputare, numărul valorilor lipsă pentru `runtime` a devenit 0, ceea ce înseamnă că setul de date poate fi folosit mai departe în analiză și modelare fără erori cauzate de valori lipsă.

In [5]:
# Eliminăm filmele fără date financiare
df_clean = df[(df['budget'] > 100000) & (df['revenue'] > 100000)].copy()
print(f" filme cu date complete: {len(df_clean)}")

# Disappointment Index
df_clean['expected_revenue'] = df_clean['budget'] * 2.5
df_clean['disappointment_index'] = df_clean['revenue'] / df_clean['expected_revenue']
df_clean['roi'] = (df_clean['revenue'] - df_clean['budget']) / df_clean['budget']
df_clean['release_year'] = pd.to_datetime(df_clean['release_date']).dt.year
df_clean['release_month'] = pd.to_datetime(df_clean['release_date']).dt.month

print(df_clean[['budget','revenue','disappointment_index','roi']].describe())

 filme cu date complete: 866
             budget       revenue  disappointment_index         roi
count  8.660000e+02  8.660000e+02            866.000000  866.000000
mean   8.310540e+07  3.606459e+08              3.005975    6.514936
std    7.746028e+07  3.805884e+08              5.531719   13.829298
min    3.250000e+05  1.243670e+05              0.000908   -0.997730
25%    2.000000e+07  8.902500e+07              0.984993    1.462482
50%    6.000000e+07  2.449706e+08              1.667067    3.167668
75%    1.300000e+08  5.047626e+08              2.809941    6.024851
max    4.899000e+08  2.923706e+09             86.474581  215.186452


### Observații după curățare

- **866 filme** au date financiare complete din 1000
- **DI mediu: 3.0** — în medie filmele câștigă de 3x față de așteptări
- **ROI mediu: 6.5** — returnul investiției e de 650% în medie
- **Std DI foarte mare (5.5)** — variații enorme între filme
- **Min ROI: -0.99** → există filme care au pierdut aproape tot bugetul

In [6]:
# Salvăm datele curate
df_clean.to_csv('../data/movies_clean.csv', index=False)
print(f" Salvat! {len(df_clean)} filme în movies_clean.csv")

 Salvat! 866 filme în movies_clean.csv
